# M5.1 · OpenVINO recovery and fair 2×2 system A/B

**Question:** Does a 331–480× faster CPU Text Encoder imply a 300× system gain? Recompute matched encoder ratios and the symmetric A/C × original/fast-client factorial from real runs. Human acceptance is scoped to C1b; INT8 numerical FAIL is never overridden.

**Learning loop:** predict → inspect frozen evidence → compute → critique → propose a controlled experiment.

Reference lesson: [open course/M05_parallel_heterogeneous/M5_1_OPENVINO_CPU_TE.md](../course/M05_parallel_heterogeneous/M5_1_OPENVINO_CPU_TE.md). Switch language: [中文](./06_openvino_and_fair_ab.zh-CN.ipynb).

**Default mode:** CPU-only, internet-free, read-only. GPU work is never run implicitly.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## Inspect and recompute real evidence

Run cells in order. Explain every denominator and scope.

In [ ]:
data = ns.read_json("runs/m51-20261105-cleanroom-01/performance/encoder_ab.json")
c0 = data["arms"]["C0_PyTorch_CPU"]["prompts"]
ov = data["arms"]["C1b_OV_BF16"]["prompts"]
for name in ("p1-zh-scene", "p2-en-scene", "p3-poster-text"):
    ratio = c0[name]["warm_median_s"] / ov[name]["warm_median_s"]
    print(f"{name}: PyTorch CPU {c0[name]['warm_median_s']:.3f}s / OV {ov[name]['warm_median_s']:.3f}s = {ratio:.1f}x")
print("These are encoder-only, positive-prompt comparisons, not E2E images/second.")


In [ ]:
study = ns.fair_factorial()
cells = study["cells"]
for name, v in cells.items():
    print(f"{name:10s} n={v['n']} mean={v['mean_s']:.3f}s => {v['images_per_min']:.3f} img/min")
gain_orig = cells["A_original"]["mean_s"] / cells["C_original"]["mean_s"]
gain_fast = cells["A_fast"]["mean_s"] / cells["C_fast"]["mean_s"]
print(f"Fair architecture gain: original client {gain_orig:.4f}x / optimized client {gain_fast:.4f}x")
assert all(v["n"] == 6 for v in cells.values())
assert 1.0 < gain_fast < 1.05
plot_values = {name: v["mean_s"] for name, v in cells.items()}
plot_ylabel, plot_title = "4-request wall seconds", "Matched 2×2 physical A/B"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


In [ ]:
registry = ns.read_json("course/data/verified_claims.json")
claims = {x["id"]: x for x in registry["claims"]}
assert claims["M51-INT8-NEGATIVE-FAIL"]["numerical_verdict"] == "FAIL"
assert claims["M51-OWNER-QUALITY-VERDICT"]["human_quality_verdict"] == "OWNER_ACCEPTED_SCOPE_ONLY"
print("C1b owner acceptance: scoped. INT8 numerical quality gate: FAIL. No status is altered.")


## Your turn / Self-check

Compare C-fast to A-original, then compare C-fast to A-fast. Why is the first an unfair architecture claim? Evaluate the trade: ~17 GB GPU VRAM freed versus ~23 GB CPU worker RSS (one host).

**Evidence discipline:** Answer in your own words. This notebook's numbers come from existing committed records, not your GPU. If your optional smoke runs, its data must remain under `runs/` and may not be presented as a full-protocol median.
